In [ ]:
from langgraph.graph import StateGraph, START, END  
# StateGraph : 그래프는 상태를 가진다는 것 (state 상태란 그래프를 통해 이동하는 데이터임)
# START, END는 예약어 같은거임 
from langgraph.pregel.main import Input
from typing_extensions import TypedDict 
from typing import Annotated
import operator



In [ ]:
# 방법1)
def update_function(old, new): 
    return  old + new 

class State(TypedDict): 
    messages :Annotated[list[str], operator.add] # 방법2) update_function 없이 이렇게 해도됨됨
   # messages :Annotated[list[str], update_function] # 방법1) 그저 langgraph에게 우리 메시지는 str list이고 이걸 update_function을 써서 업데이트 하고 싶다고 말하는 

graph_builder=StateGraph(State)

In [70]:
# 노드는 그 자리에서 state를 받을 수 있음. (당연한말)
# 아래 노드 생성(이게 우리 작업 단위임)

def node_one(state: State):
     # 마지막 메시지를 ai에게 보내고 response를 받는다
     # llm에게서 온 response로 업데이트 해야한다.  
    last_message = state["messages"][-1]
    return {"messages" :["Hello, nice to meet you!"]}
    # print("node_one ->", state) 


def node_two(state: State) -> State:
    # print("node_two ->", state)
    return {}


def node_three(state: State):
   #  print("node_three ->", state)
    return {}
   

In [71]:
# 우리는 위에 만든 node를 그래프에게 줄것임(graph_builder)

graph_builder.add_node("node_one", node_one)  # node_one이라는 이름으로 node_one함수 실행  
graph_builder.add_node("node_two", node_two) 
graph_builder.add_node("node_three", node_three)  

# node들 끼리 연결하기 위해 edge를 만든다(edge는 화살표) 
graph_builder.add_edge(START, "node_one") # start에서 "node_one"으로 화살표 추가한다는 뜻
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three") 
graph_builder.add_edge("node_three", END)
 


In [72]:
# graph = graph_builder.compile()

# graph

In [73]:
graph = graph_builder.compile() 

graph.invoke(
    {"messages": ["Hello!"]},
)

{'messages': ['Hello!', 'Hello, nice to meet you!']}